In [4]:
!pip install transformers torch fastapi uvicorn pyngrok nest-asyncio

In [5]:
!pip install --upgrade transformers[torch]

In [6]:
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import nest_asyncio
import uvicorn
from fastapi import FastAPI
from pydantic import BaseModel
from pyngrok import ngrok

In [8]:
# 1. Load the model and tokenizer directly
print("Loading model and tokenizer...")
model_name = "facebook/bart-large-cnn"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

Loading model and tokenizer...


vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

[transformers] Please make sure the generation config includes `forced_bos_token_id=0`. 


Loading weights:   0%|          | 0/511 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

In [9]:
# Move model to GPU
device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)
print(f"Model successfully loaded on: {device}")

Model successfully loaded on: cuda


In [26]:
def summarize_chunk(text_chunk):
    # Tokenize input text - no text will be lost now because chunks are small
    inputs = tokenizer(text_chunk, max_length=1024, truncation=True, return_tensors="pt").to(device)

    # Generate summary tokens with adjusted parameters for better extraction
    summary_ids = model.generate(
        inputs["input_ids"],
        max_length=200,       # Increased to allow for more points
        min_length=50,        # Forces the model to not be too brief
        length_penalty=1.5,   # Encourages slightly longer outputs
        num_beams=4,
        early_stopping=True
    )

    return tokenizer.decode(summary_ids[0], skip_special_tokens=True)

def process_large_text(text, chunk_size=500): # Reduced from 3000 to 500
    """Splits large text into smaller chunks to guarantee no truncation data loss."""
    words = text.split()

    # If the text is very short, just run it once
    if len(words) < chunk_size:
        return summarize_chunk(text)

    chunks = [" ".join(words[i:i + chunk_size]) for i in range(0, len(words), chunk_size)]

    summaries = []
    for i, chunk in enumerate(chunks):
        print(f"Summarizing section {i+1} of {len(chunks)}...")
        summaries.append(summarize_chunk(chunk))

    # Combine the dense summaries into one final document
    return "\n\n".join(summaries)

In [28]:
from fastapi import FastAPI
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel

app = FastAPI()

# --- ADD THIS CORS BLOCK ---
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"], # Allows your Chrome Extension to connect
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)
# ---------------------------

class Payload(BaseModel):
    text: str

@app.post("/summarize")
def handle_summarize(payload: Payload):
    if not payload.text.strip():
        return {"summary": "No text provided to summarize."}

    final_summary = process_large_text(payload.text)
    return {"summary": final_summary}

In [29]:
from google.colab import userdata
from pyngrok import ngrok

In [30]:
try:
    NGROK_TOKEN = userdata.get('NGROK_AUTHTOKEN')
    ngrok.set_auth_token(NGROK_TOKEN)
    print("✓ Ngrok token authenticated securely.")
except Exception as e:
    print("⚠️ Could not find NGROK_AUTHTOKEN in Colab Secrets.")

✓ Ngrok token authenticated securely.


In [31]:
public_url = ngrok.connect(8000)
print(f"👉 COPY THIS URL FOR CHROME EXTENSION: {public_url.public_url}")

👉 COPY THIS URL FOR CHROME EXTENSION: https://selene-goateed-overprovidently.ngrok-free.dev


In [32]:
import asyncio
import uvicorn

# 1. Create a configuration object for Uvicorn
config = uvicorn.Config(app, host="0.0.0.0", port=8000)

# 2. Initialize the server with that config
server = uvicorn.Server(config)

# 3. 'await' the server to run it on Colab's existing event loop
await server.serve()

INFO:     Started server process [3180]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


Summarizing section 1 of 2...
Summarizing section 2 of 2...
INFO:     2402:d000:8134:936:3147:c2e7:1115:91e7:0 - "POST /summarize HTTP/1.1" 200 OK


INFO:     Shutting down
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.
INFO:     Finished server process [3180]
